# KNN - Classificação

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pcj.utils import BASE_DIR, dados

In [ ]:
# Dados para testes

dados_limpos, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")

X = dados_limpos.copy()

In [ ]:
# Verificações

# print(dados_limpos['Data'].dtype, end='')
# print(dados_limpos['data_normalizada'].dtype)
# print(dados_limpos.columns)
# print(variaveis)
print(dados_limpos[variaveis])

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 01 - Seguindo tutorial 

Link para o tutorial: https://medium.com/@ulissesmaffa/machine-learning-algoritmo-knn-26eb7b702c37

Diferenças entre o tutorial e o código nesse notebook:
- "x" substituído por "dados_limpos";
- "y" substituído por "colunas";
- Utilização de `KNeighborsRegressor` ao invés de `KNeighborsClassifier`, 
uma vez que não há dados categórigos na base de interesse;
- Utilização das métricas "RMSE", "MAE", e "r2" ao invés da matriz de confusão,
uma vez que matrizes de confusão funcionam para dados categóricos. As métricas citadas
funcionam com dados contínuos.

In [ ]:
# Converte os títulos das colunas em uma Series do pandas
colunas = pd.Series(variaveis)

target_col = colunas[0] # Seleciona uma coluna (variável) alvo

In [ ]:
# Normalizção dos dados

from sklearn.preprocessing import MinMaxScaler

normalize = MinMaxScaler(feature_range=(0,1))

limpos_norm = normalize.fit_transform(dados_limpos[variaveis])

limpos_norm = pd.DataFrame(limpos_norm, columns=colunas)

In [ ]:
y = limpos_norm[target_col]

x = limpos_norm.drop(target_col, axis=1)    # Features = Todas as colunas, exceto a alvo

In [ ]:
# Divisão em conjuntos de treinamento e teste para treinamento

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [ ]:
# Inicialização do modelo

# from sklearn.neighbors import KNeighborsClassifier

from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=5)

# Treinamento

model.fit(x_train, y_train)

In [ ]:
# Avaliação do Modelo com o método `score`

result = model.score(x_test, y_test)

print(f"Precisão do modelo no conjunto de teste: {result: .2%}")

In [ ]:
# Ajuste dos Hiperparâmetros com GridSearchCV

## Definição da grade de hiperparâmetros

param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15], # Número de vizinhos considerados pelo algoritmo
    'weights': ['uniform', 'distance'], # Como o algoritmo deve pesar as distâncias dos vizinhos
                                        # 'uniform' = uniformemente, 'distance' = em função da distância entre vizinhos
    'metric': ['euclidean', 'manhattan', 'minkowski']   # Que métrica de distância usar
}

In [ ]:
# Executando o GridSearchCV

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(KNeighborsRegressor(), param_grid,
                            cv=5)   # Coloquei 5 por causa do tutorial, não sei do que se trata

grid_search.fit(x_train, y_train)

In [ ]:
# Resultados

print("Melhores parâmetros: ", grid_search.best_params_)

In [ ]:
# Treinamento com os Melhores Parâmetros

## Seleção dos melhores hiperparâmetros identificados pelo GridSearchCV

best_knn = KNeighborsRegressor(**grid_search.best_params_)

## Treinamento

best_knn.fit(x_train, y_train)

In [ ]:
# Avaliação do modelo com Matriz de Confusão

# from sklearn.metrics import confusion_matrix,
# sns.heatmap(conf_matrix, annot=True, fmt='g')   # Estou seguindo o tutorial, não entendo isso direito
# plt.xlabel('Previsão')
# plt.ylabel('Real')
# plt.show()

# Avaliação do modelo com Métricas

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

y_pred = best_knn.predict(x_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")


Automatizando o processo para verificar todas as variáveis

In [ ]:
# Set up
# 1) Bibliotecas

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 2) Seleção de variável e normalização de colunas

colunas = pd.Series(variaveis)

normalize = MinMaxScaler(feature_range=(0,1))

limpos_norm = normalize.fit_transform(dados_limpos[variaveis])

limpos_norm = pd.DataFrame(limpos_norm, columns=colunas)

# 3) Loop

# Selecionará cada variável como variável alvo
for target_col in colunas:
    print(f"Variável: {target_col}", "\n")

    y = limpos_norm[target_col]
    x = limpos_norm.drop(target_col, axis=1)    # Features = Todas as colunas, exceto a alvo
    
    # Divisão em conjuntos de treinamento e teste para treinamento
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

    # Inicialização do modelo
    model = KNeighborsRegressor(n_neighbors=5)

    # Treinamento
    model.fit(x_train, y_train)

    # Avaliação do Modelo com o método `score`
    result = model.score(x_test, y_test)
    print(f"Precisão do modelo no conjunto de teste: {result: .2%}")

    # Ajuste dos Hiperparâmetros com GridSearchCV
    ## Definição da grade de hiperparâmetros
    param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15], # Número de vizinhos considerados pelo algoritmo
    'weights': ['uniform', 'distance'], # Como o algoritmo deve pesar as distâncias dos vizinhos
                                        # 'uniform' = uniformemente, 'distance' = em função da distância entre vizinhos
    'metric': ['euclidean', 'manhattan', 'minkowski']   # Que métrica de distância usar
    }

    # Executando o GridSearchCV
    grid_search = GridSearchCV(KNeighborsRegressor(), param_grid,
                            cv=5)   # Coloquei 5 por causa do tutorial, não sei do que sse trata

    grid_search.fit(x_train, y_train)

    ## Resultados

    print("Melhores parâmetros: ", grid_search.best_params_)
    
    # Treinamento com os Melhores Parâmetros
    ## Seleção dos melhores hiperparâmetros identificados pelo GridSearchCV
    best_knn = KNeighborsRegressor(**grid_search.best_params_)

    ## Treinamento
    best_knn.fit(x_train, y_train)

    # Avaliação do modelo com Métricas
    y_pred = best_knn.predict(x_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}")
    print("--------------")


![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

Provavelmente seria interessante adicionar algumas visualizações aqui